# ARecords with ECH Config, Grouped by ASN

In [ ]:
import os
import pickle
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.pool import QueuePool

# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location

testDates = [
    "2024-08-08",
    "2024-08-19",
    "2024-08-21",
    "2024-08-26",
    "2024-08-29",
    "2024-09-01",
    "2024-09-05",
    "2024-09-06",
    "2024-09-08",
    "2024-09-11",
    "2024-09-15",
    "2024-10-13",
    "2024-10-17",
    "2024-10-20",
    "2024-10-27",
    "2024-10-29",
    "2024-10-31",
    "2024-11-07",
    "2024-11-12",
]


def fetch_and_save_data(query, filename_suffix, params=None, timeout=3600):
    """
    Fetches data from PostgreSQL using SQLAlchemy with connection pooling,
    saves it to a pickle file, and handles timeouts.
    Reads database credentials from environment variables.

    Args:
        query (str): The SQL query to execute.
        filename_suffix (str): A string to be added to the pickle filename.
        params (tuple, optional): Parameters to pass to the query. Defaults to None.
        timeout (int, optional): The timeout for the database connection in seconds. Defaults to 3600.
    """
    try:
        db_user = os.getenv("DB_USER")
        db_password = os.getenv("DB_PASSWORD")
        db_host = os.getenv("DB_HOST")
        db_name = os.getenv("DB_NAME")

        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}/{db_name}",
            connect_args={"connect_timeout": timeout},
            poolclass=QueuePool,
            pool_size=5,
            max_overflow=10,
        )

        # Create an empty list to store the dataframes
        df_list = []

        with engine.connect() as conn:
            # Iterate over the testDates list
            for date in testDates:
                # Execute the query for each date
                params = (date,)
                df = pd.read_sql_query(query, conn, params=params)
                df_list.append(df)

        # Concatenate all the dataframes into a single dataframe
        final_df = pd.concat(df_list, ignore_index=True)

        print(final_df)
        os.makedirs("./pickle", exist_ok=True)
        filename = f"./pickle/{datetime.now().strftime('%Y%m%d_%H%M%S')}_{filename_suffix}.pickle"

        with open(filename, "wb") as f:
            pickle.dump(final_df, f)

        print(f"Data saved to {filename}")

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    query = """
SELECT 
    hr.test_date,
    ar.asn_org, 
    COUNT(DISTINCT hr.domain) AS domain_count
FROM 
    public."DNSResults" dr
JOIN 
    public."DNSResultsARecords" dra ON dr.id = dra.dns_result_id
JOIN 
    public."ARecords" ar ON dra.a_record_id = ar.id
JOIN 
    public."DNSResultsHTTPSRecords" drhr ON dr.id = drhr.dns_result_id
JOIN 
    public."HTTPSRecords" hr ON drhr.https_record_id = hr.id
WHERE 
    hr.ech_config IS NOT NULL
    AND hr.ech_config != ''
    AND hr.test_date = %s 
GROUP BY 
    hr.test_date, ar.asn_org
ORDER BY 
    hr.test_date, domain_count DESC;
    """
    filename_suffix = "ech_a_asn_test_dates"
    fetch_and_save_data(query, filename_suffix)

# AAAARecords with ECH Config, Grouped by ASN

In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location

testDates = [
    "2024-08-08",
    "2024-08-19",
    "2024-08-21",
    "2024-08-26",
    "2024-08-29",
    "2024-09-01",
    "2024-09-05",
    "2024-09-06",
    "2024-09-08",
    "2024-09-11",
    "2024-09-15",
    "2024-10-13",
    "2024-10-17",
    "2024-10-20",
    "2024-10-27",
    "2024-10-29",
    "2024-10-31",
    "2024-11-07",
    "2024-11-12",
]


def fetch_and_save_data(query, filename_suffix, params=None, timeout=3600):
    """
    Fetches data from PostgreSQL using SQLAlchemy with connection pooling,
    saves it to a pickle file, and handles timeouts.
    Reads database credentials from environment variables.

    Args:
        query (str): The SQL query to execute.
        filename_suffix (str): A string to be added to the pickle filename.
        params (tuple, optional): Parameters to pass to the query. Defaults to None.
        timeout (int, optional): The timeout for the database connection in seconds. Defaults to 3600.
    """
    try:
        db_user = os.getenv("DB_USER")
        db_password = os.getenv("DB_PASSWORD")
        db_host = os.getenv("DB_HOST")
        db_name = os.getenv("DB_NAME")

        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}/{db_name}",
            connect_args={"connect_timeout": timeout},
            poolclass=QueuePool,
            pool_size=5,
            max_overflow=10,
        )

        # Create an empty list to store the dataframes
        df_list = []

        with engine.connect() as conn:
            # Iterate over the testDates list
            for date in testDates:
                # Execute the query for each date
                params = (date,)
                df = pd.read_sql_query(query, conn, params=params)
                df_list.append(df)

        # Concatenate all the dataframes into a single dataframe
        final_df = pd.concat(df_list, ignore_index=True)

        print(final_df)
        os.makedirs("./pickle", exist_ok=True)
        filename = f"./pickle/{datetime.now().strftime('%Y%m%d_%H%M%S')}_{filename_suffix}.pickle"

        with open(filename, "wb") as f:
            pickle.dump(final_df, f)

        print(f"Data saved to {filename}")

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    query = """
SELECT 
    hr.test_date,
    ar.asn_org, 
    COUNT(DISTINCT hr.domain) AS domain_count
FROM 
    public."DNSResults" dr
JOIN 
    public."DNSResultsAAAARecords" dra ON dr.id = dra.dns_result_id
JOIN 
    public."AAAARecords" ar ON dra.aaaa_record_id = ar.id
JOIN 
    public."DNSResultsHTTPSRecords" drhr ON dr.id = drhr.dns_result_id
JOIN 
    public."HTTPSRecords" hr ON drhr.https_record_id = hr.id
WHERE 
    hr.ech_config IS NOT NULL
    AND hr.ech_config != ''
    AND hr.test_date = %s
GROUP BY 
    hr.test_date, ar.asn_org
ORDER BY 
    hr.test_date, domain_count DESC;
    """
    filename_suffix = "ech_aaaa_asn_test_dates"
    fetch_and_save_data(query, filename_suffix)

# Authoritative NS with ECH Config, Grouped by ASN

In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location

testDates = [
    "2024-08-08",
    "2024-08-19",
    "2024-08-21",
    "2024-08-26",
    "2024-08-29",
    "2024-09-01",
    "2024-09-05",
    "2024-09-06",
    "2024-09-08",
    "2024-09-11",
    "2024-09-15",
    "2024-10-13",
    "2024-10-17",
    "2024-10-20",
    "2024-10-27",
    "2024-10-29",
    "2024-10-31",
    "2024-11-07",
    "2024-11-12",
]


def fetch_and_save_data(query, filename_suffix, params=None, timeout=3600):
    """
    Fetches data from PostgreSQL using SQLAlchemy with connection pooling,
    saves it to a pickle file, and handles timeouts.
    Reads database credentials from environment variables.

    Args:
        query (str): The SQL query to execute.
        filename_suffix (str): A string to be added to the pickle filename.
        params (tuple, optional): Parameters to pass to the query. Defaults to None.
        timeout (int, optional): The timeout for the database connection in seconds. Defaults to 3600.
    """
    try:
        db_user = os.getenv("DB_USER")
        db_password = os.getenv("DB_PASSWORD")
        db_host = os.getenv("DB_HOST")
        db_name = os.getenv("DB_NAME")

        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}/{db_name}",
            connect_args={"connect_timeout": timeout},
            poolclass=QueuePool,
            pool_size=5,
            max_overflow=10,
        )

        # Create an empty list to store the dataframes
        df_list = []

        with engine.connect() as conn:
            # Iterate over the testDates list
            for date in testDates:
                # Execute the query for each date
                params = (date,)
                df = pd.read_sql_query(query, conn, params=params)
                df_list.append(df)

        # Concatenate all the dataframes into a single dataframe
        final_df = pd.concat(df_list, ignore_index=True)

        print(final_df)
        os.makedirs("./pickle", exist_ok=True)
        filename = f"./pickle/{datetime.now().strftime('%Y%m%d_%H%M%S')}_{filename_suffix}.pickle"

        with open(filename, "wb") as f:
            pickle.dump(final_df, f)

        print(f"Data saved to {filename}")

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    query = """
SELECT 
    hr.test_date,
    ar.asn_org, 
    COUNT(DISTINCT hr.domain) AS domain_count
FROM 
    public."DNSResults" dr
JOIN 
    public."DNSResultsHTTPSRecords" drhr ON dr.id = drhr.dns_result_id
JOIN 
    public."HTTPSRecords" hr ON drhr.https_record_id = hr.id
JOIN 
    public."DNSResultsAuthoritativeNameserverRecords" dra ON dr.id = dra.dns_result_id
JOIN 
    public."AuthoritativeNameserverRecords" ar ON dra.authoritative_nameserver_record_id = ar.id
WHERE 
    hr.ech_config IS NOT NULL
    AND hr.ech_config != ''
    AND hr.test_date = %s
GROUP BY 
    hr.test_date, ar.asn_org
ORDER BY 
    hr.test_date, domain_count DESC;
    """
    filename_suffix = "ech_authoritative-ns_asn_test_dates"
    fetch_and_save_data(query, filename_suffix)


# Single TestCode relation of Authoritative NS and ECH Config
SELECT ar.asn_org, COUNT(DISTINCT dr.domain) AS domain_count
FROM public."DNSResults" dr
JOIN public."DNSResultsAuthoritativeNameservers" dra ON dr.id = dra.dns_result_id
JOIN public."AuthoritativeNameservers" ar ON dra.authoritative_nameserver_record_id = ar.id
WHERE dr.test_code = '2024-10-13_02-50-30_8.8.8.8'
  AND dr.https_ech_key IS NOT NULL
  AND dr.https_ech_key <> ''
GROUP BY ar.asn_org
ORDER BY domain_count DESC;